# 🔬 Verificación del Entorno — Laboratorio TFM

## Evaluación de Ciberseguridad en Agentes de IA Autónomos

**Trabajo de Fin de Máster** | Máster en Ciberseguridad  
**Framework**: OpenCode + Ollama + Gemma 4  
**Objetivo del notebook**: Verificar que el entorno del laboratorio está correctamente configurado antes de ejecutar los experimentos.

---

### 🏗️ Arquitectura del laboratorio

```
┌─────────────────────────────────────────────────┐
│  Docker Container (Jupyter)                     │
│  ┌───────────────────────────────────────────┐  │
│  │  Notebooks de Experimentos               │  │
│  │  /home/jovyan/work/                      │  │
│  └───────────────────────────────────────────┘  │
│           │ HTTP requests                        │
└───────────┼─────────────────────────────────────┘
            │ host.docker.internal:11434
┌───────────▼─────────────────────────────────────┐
│  Host OS (Windows)                              │
│  Ollama Server → Gemma 4 (e2b / e4b / 26b)     │
└─────────────────────────────────────────────────┘
```

### 📋 Vectores de ataque del laboratorio

| Notebook | Vector | OWASP LLM |
|----------|--------|----------|
| 01 | Prompt Injection Directa | LLM01:2025 |
| 02 | Prompt Injection Indirecta | LLM02:2025 |
| 03 | Jailbreaking | LLM01:2025 |
| 04 | Comparativa de Modelos | — |

---

> ⚠️ **AVISO ÉTICO**: Todos los experimentos de este laboratorio se realizan en un entorno controlado con el único propósito de investigación académica. Los prompts de ataque no deben usarse fuera de este contexto.

In [ ]:
# ============================================================
# CELDA 1: Verificar conexión con Ollama y listar modelos
# ============================================================

import requests
import json
import time
from pathlib import Path
from datetime import datetime

# --- Detección automática del endpoint ---
CANDIDATES = [
    "http://host.docker.internal:11434",  # Dentro de Docker
    "http://localhost:11434",             # Ejecución local / WSL
    "http://127.0.0.1:11434",            # Alternativa local
]

OLLAMA_URL = None
for candidate in CANDIDATES:
    try:
        r = requests.get(f"{candidate}/api/tags", timeout=5)
        if r.status_code == 200:
            OLLAMA_URL = candidate
            print(f"✅ Ollama encontrado en: {OLLAMA_URL}")
            break
    except Exception:
        pass

if OLLAMA_URL is None:
    raise ConnectionError(
        "❌ No se puede conectar con Ollama. "
        "Asegúrate de que el servicio está activo con: ollama serve"
    )

# --- Listar modelos disponibles ---
response = requests.get(f"{OLLAMA_URL}/api/tags", timeout=10)
models_data = response.json()
available_models = [m["name"] for m in models_data.get("models", [])]

print(f"\n📦 Modelos disponibles en Ollama ({len(available_models)} total):")
for m in available_models:
    size_info = next((x for x in models_data["models"] if x["name"] == m), {})
    size_gb = size_info.get("size", 0) / 1e9
    print(f"   • {m:<30} ({size_gb:.1f} GB)")

# --- Verificar modelos Gemma 4 del TFM ---
TFM_MODELS = ["gemma4:e2b", "gemma4:e4b", "gemma4:26b"]
print("\n🎯 Estado de modelos del TFM:")
for model in TFM_MODELS:
    found = any(model in m for m in available_models)
    status = "✅ DISPONIBLE" if found else "❌ NO DESCARGADO"
    print(f"   {model:<20} → {status}")

In [ ]:
# ============================================================
# CELDA 2: Verificar que num_ctx=127000 funciona correctamente
# ============================================================

# Seleccionar el modelo disponible para el test
test_model = None
for tm in TFM_MODELS:
    if any(tm in m for m in available_models):
        test_model = next(m for m in available_models if tm in m)
        break

# Si no hay modelos Gemma 4, usar el primero disponible
if test_model is None and available_models:
    test_model = available_models[0]
    print(f"⚠️  No hay modelos Gemma 4 disponibles. Usando: {test_model}")
elif test_model is None:
    raise RuntimeError("❌ No hay modelos disponibles en Ollama.")

print(f"🔧 Probando modelo: {test_model}")
print(f"🔧 Configuración: num_ctx=127000 (ventana de contexto máxima de Gemma 4)\n")

# Test de chat con num_ctx extendido
payload = {
    "model": test_model,
    "messages": [
        {"role": "user", "content": "Responde en español con exactamente estas palabras: 'Sistema operativo correctamente configurado.'"}
    ],
    "stream": False,
    "options": {"num_ctx": 127000}
}

start_time = time.time()
try:
    r = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=120)
    latency_ms = int((time.time() - start_time) * 1000)
    
    if r.status_code == 200:
        content = r.json()["message"]["content"]
        print(f"✅ Chat funciona correctamente con num_ctx=127000")
        print(f"📝 Respuesta del modelo: '{content.strip()}'")
        print(f"⏱️  Latencia: {latency_ms} ms")
    else:
        print(f"❌ Error HTTP {r.status_code}: {r.text}")
except requests.exceptions.Timeout:
    print("⏰ Timeout — el modelo tardó más de 120 segundos. Prueba con un modelo más pequeño.")
except Exception as e:
    print(f"❌ Error inesperado: {e}")

In [ ]:
# ============================================================
# CELDA 3: Función helper chat() — se importará en el resto
# de notebooks o se copia directamente
# ============================================================

def chat(model, messages, system=None, num_ctx=127000, timeout=120):
    """
    Función principal para interactuar con Ollama.
    
    Parámetros:
        model    : nombre del modelo Ollama (ej: 'gemma4:e2b')
        messages : lista de dicts [{"role": "user", "content": "..."}]
        system   : prompt de sistema opcional (str)
        num_ctx  : tamaño del contexto (default 127000 para Gemma 4)
        timeout  : timeout en segundos
    
    Retorna:
        (content: str, latency_ms: int)
    """
    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {"num_ctx": num_ctx}
    }
    
    # Añadir system prompt si se especifica
    if system:
        payload["messages"] = [
            {"role": "system", "content": system}
        ] + messages
    
    start = time.time()
    try:
        r = requests.post(
            f"{OLLAMA_URL}/api/chat",
            json=payload,
            timeout=timeout
        )
        latency_ms = int((time.time() - start) * 1000)
        r.raise_for_status()
        content = r.json()["message"]["content"]
        return content, latency_ms
    except requests.exceptions.Timeout:
        return "[ERROR: Timeout]", -1
    except Exception as e:
        return f"[ERROR: {str(e)}]", -1


# --- Test rápido de la función ---
print("🧪 Testeando la función helper chat()...")
response, latency = chat(
    model=test_model,
    messages=[{"role": "user", "content": "Di solo 'OK' sin nada más."}],
    system="Eres un asistente que responde con una sola palabra."
)
print(f"✅ Función chat() operativa")
print(f"   Modelo  : {test_model}")
print(f"   Respuesta: {response.strip()}")
print(f"   Latencia : {latency} ms")

## 📥 Cómo descargar los modelos si no están disponibles

Si alguno de los modelos Gemma 4 no aparece como disponible, ejecúta los siguientes comandos en el **host** (no en Docker):

```bash
# Modelo pequeño — 2 billones de parámetros (~2 GB)
ollama pull gemma4:e2b

# Modelo mediano — 4 billones de parámetros (~4 GB)
ollama pull gemma4:e4b

# Modelo grande — 26 billones de parámetros (~16 GB)
ollama pull gemma4:26b
```

### Requisitos de hardware recomendados:

| Modelo | VRAM mínima | RAM mínima | Tiempo est. de descarga |
|--------|-------------|------------|------------------------|
| gemma4:e2b | 4 GB | 8 GB | ~5 min |
| gemma4:e4b | 6 GB | 12 GB | ~10 min |
| gemma4:26b | 16 GB | 32 GB | ~30 min |

### Verificar que Ollama está activo:

```bash
# En Windows PowerShell
ollama serve

# O verificar el proceso
Get-Process ollama
```

> 💡 **Tip**: Para los experimentos básicos basta con `gemma4:e2b`. El notebook `04_comparativa_modelos.ipynb` ejecutará automáticamente sólo los modelos disponibles.

In [ ]:
# ============================================================
# CELDA 4: Guardar configuración del laboratorio (LAB_CONFIG)
# ============================================================

import uuid

# Determinar el modelo por defecto (el primero disponible de la lista de prioridad)
DEFAULT_MODEL = None
for preferred in TFM_MODELS:
    match = next((m for m in available_models if preferred in m), None)
    if match:
        DEFAULT_MODEL = match
        break

if DEFAULT_MODEL is None and available_models:
    DEFAULT_MODEL = available_models[0]

# Crear directorio de resultados
results_dir = Path("/home/jovyan/work/lab/results")
results_dir.mkdir(parents=True, exist_ok=True)

# Crear directorio alternativo para entornos locales (fuera de Docker)
local_results = Path("../lab/results")
local_results.mkdir(parents=True, exist_ok=True)

# Configuración global del laboratorio
LAB_CONFIG = {
    "session_id": str(uuid.uuid4()),
    "timestamp": datetime.now().isoformat(),
    "ollama_url": OLLAMA_URL,
    "default_model": DEFAULT_MODEL,
    "available_models": available_models,
    "tfm_models_available": {
        model: any(model in m for m in available_models)
        for model in TFM_MODELS
    },
    "num_ctx": 127000,
    "results_dir": str(results_dir),
    "lab_vectors": [
        "direct_injection",
        "indirect_injection",
        "jailbreak",
        "model_comparison"
    ]
}

# Guardar config como JSON
config_path = results_dir / "lab_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(LAB_CONFIG, f, indent=2, ensure_ascii=False)

# También guardar en ruta local
local_config = local_results / "lab_config.json"
with open(local_config, "w", encoding="utf-8") as f:
    json.dump(LAB_CONFIG, f, indent=2, ensure_ascii=False)

# Mostrar resumen
print("=" * 60)
print("📊 CONFIGURACIÓN DEL LABORATORIO TFM")
print("=" * 60)
print(f"🆔 Session ID   : {LAB_CONFIG['session_id']}")
print(f"🕐 Timestamp    : {LAB_CONFIG['timestamp']}")
print(f"🌐 Ollama URL   : {LAB_CONFIG['ollama_url']}")
print(f"🤖 Modelo default: {LAB_CONFIG['default_model']}")
print(f"📁 Resultados   : {LAB_CONFIG['results_dir']}")
print()
print("🎯 Modelos TFM disponibles:")
for model, available in LAB_CONFIG["tfm_models_available"].items():
    icon = "✅" if available else "❌"
    print(f"   {icon} {model}")
print()
print(f"💾 Config guardada en: {config_path}")
print("=" * 60)
print("✅ ENTORNO VERIFICADO — El laboratorio está listo para ejecutar experimentos")
print("=" * 60)